In [3]:
from rdkit.Chem import Descriptors, MolFromSmiles
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, RepeatedKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, root_mean_squared_error
from pandas import DataFrame, concat
from pickle import load
from tqdm import tqdm

In [4]:
#создаем словарь из дескприторов структуры
ConstDescriptors = {"HeavyAtomCount" : Descriptors.HeavyAtomCount,
                       "NHOHCount" : Descriptors.NHOHCount,
                       "NOCount" : Descriptors.NOCount,
                       "NumHAcceptors" : Descriptors.NumHAcceptors,
                       "NumHDonors" : Descriptors.NumHDonors,
                       "NumHeteroatoms" : Descriptors.NumHeteroatoms,
                       "NumRotatableBonds" : Descriptors.NumRotatableBonds,
                       "NumValenceElectrons" : Descriptors.NumValenceElectrons,
                       "NumAromaticRings" : Descriptors.NumAromaticRings,
                       "NumAliphaticHeterocycles" : Descriptors.NumAliphaticHeterocycles,
                       "RingCount" : Descriptors.RingCount}

#создаем словарь из физико-химических дескрипторов                            
PhisChemDescriptors = {"MW" : Descriptors.MolWt,
                          "LogP" : Descriptors.MolLogP,
                          "MR" : Descriptors.MolMR,
                          "TPSA" : Descriptors.TPSA}

#объединяем все дескрипторы в один словарь
descriptors = {}
descriptors.update(ConstDescriptors)
descriptors.update(PhisChemDescriptors)
print(len(descriptors), "количество дескрипторов в словаре")


#функция для генерации дескрипторов из молекул
def mol_dsc_calc(mols):
    
    return DataFrame({k: f(m) for k, f in descriptors.items()} 
             for m in mols)


with open('../data/classifier/no_dubl_potency.pickle', 'rb') as file:
    potency = load(file)
#оформляем sklearn трансформер для использования в конвеерном моделировании (sklearn Pipeline)
descriptors_transformer = FunctionTransformer(mol_dsc_calc, validate=False)

15 количество дескрипторов в словаре


In [5]:
molecules = [
    MolFromSmiles(mol) for mol in potency['SMILES']
]

In [6]:
X = descriptors_transformer.transform(molecules)
Y = potency['Potency']
XY = concat((X, Y), axis=1)

In [7]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.15)

In [13]:
scaler_x = StandardScaler().fit(x_train)
x_train_scal = scaler_x.transform(x_train)
x_test_scal = scaler_x.transform(x_test)

In [14]:
scaler_y = MinMaxScaler().fit(y_train.to_numpy().reshape(-1, 1))
y_train_scal = scaler_y.transform(y_train.to_numpy().reshape(-1, 1))
y_test_scal = scaler_y.transform(y_test.to_numpy().reshape(-1, 1))

In [ ]:
rf = RandomForestRegressor()
pg = {'n_estimators': [50, 100, 200, 300, 400, 500],
      'max_depth': [None, 1, 3, 5, 7, 10],
      }
cv = RepeatedKFold(n_splits=5, n_repeats=5)
gs = GridSearchCV(rf, pg, verbose=3, cv=cv)

gs.fit(x_train_scal, y_train_scal.ravel())

Fitting 25 folds for each of 36 candidates, totalling 900 fits
[CV 1/25] END ..max_depth=None, n_estimators=50;, score=0.144 total time=   2.5s
[CV 2/25] END ..max_depth=None, n_estimators=50;, score=0.182 total time=   2.3s
[CV 3/25] END ..max_depth=None, n_estimators=50;, score=0.140 total time=   2.3s
[CV 4/25] END ..max_depth=None, n_estimators=50;, score=0.125 total time=   2.3s
[CV 5/25] END ..max_depth=None, n_estimators=50;, score=0.169 total time=   2.3s
[CV 6/25] END ..max_depth=None, n_estimators=50;, score=0.168 total time=   2.3s
[CV 7/25] END ..max_depth=None, n_estimators=50;, score=0.139 total time=   2.3s
[CV 8/25] END ..max_depth=None, n_estimators=50;, score=0.157 total time=   2.3s
[CV 9/25] END ..max_depth=None, n_estimators=50;, score=0.136 total time=   2.3s
[CV 10/25] END .max_depth=None, n_estimators=50;, score=0.132 total time=   2.3s
[CV 11/25] END .max_depth=None, n_estimators=50;, score=0.129 total time=   2.3s
[CV 12/25] END .max_depth=None, n_estimators=5

In [ ]:
gs.best_estimator_

In [ ]:
y_pred = gs.predict(x_test_scal)
print(f'R^2 = {r2_score(y_test_scal, y_pred)}')
print(f'RMSE = {root_mean_squared_error(y_test_scal, y_pred)}')